
# Projenin ve Analizin Amacı
Bu analiz fraudshield-ml projesinin ilk adımı olarak **Veri Doğrulama ve Keşifsel Veri Analizi (EDA)** kısımlarını içermektedir.
Amacımız, makine öğrenmesi modelleri kurmadan önce verinin kalitesini, özelliklerini, hedef değişken dengesizliğini ve çeşitli anormallikleri saptamaktır.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Ayarlar
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Src modüllerini yola ekleme
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.data.load_data import load_raw_data
from src.data.validate_data import validate_raw_data



## Verinin Okunması
Veri, modüler olarak hazırlanmış `src.data.load_data` içerisindeki fonksiyon kullanılarak bellek optimizasyonu ile yüklenmektedir.


In [ ]:

DATA_PATH = '../data/raw/PS_20174392719_1491204439457_log.csv'
try:
    df = load_raw_data(DATA_PATH)
    print("Veri başarıyla yüklendi.")
except Exception as e:
    print(f"Hata: {e}")



## Genel Veri Görünümü ve Kalite Kontrolleri
Gerekli veri doğrulama işlemleri (eksik değerler, negatif miktarlar, tekrar eden kayıtlar vs.) `src.data.validate_data` ile gerçekleştirilecektir.


In [ ]:

if 'df' in locals():
    validation_results = validate_raw_data(df)
    import pprint
    pprint.pprint(validation_results)
    
    # Genel görünüm
    display(df.head())
    display(df.describe())



## Hedef Değişken (isFraud) Dağılımı
Sınıf dengesizliği yüksek olacağından logaritmik ölçekte göstermek daha iyi bir fikir verebilir.


In [ ]:

if 'df' in locals():
    plt.figure(figsize=(8,5))
    ax = sns.countplot(data=df, x='isFraud')
    ax.set_yscale('log')
    plt.title('isFraud Dağılımı (Log Ölçek)')
    plt.ylabel('Kayıt Sayısı (Log)')
    plt.xlabel('Dolandırıcılık Durumu (0: Normal, 1: Fraud)')
    for p in ax.patches:
        ax.annotate(f'{p.get_height()}', (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='bottom', fontsize=10, color='black', xytext=(0, 5),
                    textcoords='offset points')
    plt.show()



## İşlem Türlerine (type) Göre Dağılım ve Fraud Analizi
Hangi işlem türleri daha çok kullanılıyor ve hangi işlem türlerinde fraud görülüyor?


In [ ]:

if 'df' in locals():
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    sns.countplot(data=df, x='type', ax=axes[0], order=df['type'].value_counts().index)
    axes[0].set_yscale('log')
    axes[0].set_title('İşlem Türlerinin Dağılımı (Log Ölçek)')
    axes[0].set_ylabel('İşlem Sayısı (Log)')
    
    fraud_df = df[df['isFraud'] == 1]
    sns.countplot(data=fraud_df, x='type', ax=axes[1], order=df['type'].value_counts().index)
    axes[1].set_title('Yalnızca Fraud İşlemlerinin Türlere Göre Dağılımı')
    axes[1].set_ylabel('Fraud İşlem Sayısı')
    
    plt.tight_layout()
    plt.show()


In [ ]:

if 'df' in locals():
    fraud_counts = fraud_df['type'].value_counts()
    total_counts = df['type'].value_counts()
    
    fraud_rates = (fraud_counts / total_counts).dropna() * 100
    
    plt.figure(figsize=(8,5))
    sns.barplot(x=fraud_rates.index, y=fraud_rates.values)
    plt.title('İşlem Türlerine Göre Fraud Oranı (%)')
    plt.ylabel('Fraud Oranı (%)')
    plt.xlabel('İşlem Türü')
    plt.show()



## Tutar (Amount) ve Bakiye Analizi
Fraud ve normal işlemlerin işlem tutarlarının (amount) dağılımı nasıl değişiyor?
Burada verinin bir kısmını (örneklem) kullanmak görselleştirme performansını artırır.


In [ ]:

if 'df' in locals():
    df_sample = df.sample(n=min(len(df), 500000), random_state=42)
    
    plt.figure(figsize=(10,6))
    sns.boxplot(data=df_sample, x='isFraud', y='amount')
    plt.yscale('log')
    plt.title('Fraud ve Normal İşlemlerin Tutar Dağılımı (Log Ölçek, Örneklem)')
    plt.show()



## Fraud İşlemlerinin Step Değerlerine Göre Dağılımı
`step` değişkeni her 1 saati ifade eder. Fraud işlemlerinde zamana bağlı bir örüntü var mı?


In [ ]:

if 'df' in locals():
    plt.figure(figsize=(14,6))
    sns.histplot(data=df[df['isFraud']==1], x='step', bins=100, kde=True, color='red', label='Fraud')
    sns.histplot(data=df_sample[df_sample['isFraud']==0], x='step', bins=100, color='blue', alpha=0.3, label='Normal (Örneklem)')
    plt.title('Step (Zaman) Dağılımı')
    plt.legend()
    plt.show()



## isFlaggedFraud Analizleri
Mevcut kurallar fraud'ları ne kadar iyi yakalıyor?


In [ ]:

if 'df' in locals():
    print("isFraud ve isFlaggedFraud Karşılaştırması:\n")
    print(pd.crosstab(df['isFraud'], df['isFlaggedFraud']))



## İlk Bulgular ve Sonraki Aşama Önerileri

**Bulgular:**
- Veride çok ciddi bir sınıf dengesizliği bulunmaktadır.
- Fraud'lar sadece belirli işlem türlerinde gözlemlenmektedir.
- `isFlaggedFraud` değişkeni neredeyse hiçbir fraud işlemini doğru işaretleyememiştir.
- Tutar (amount) bazında incelendiğinde, fraud işlemlerin yüksek tutarlı olması muhtemeldir.

**Öneriler:**
- Makine öğrenmesi aşamasına geçerken **SMOTE** veya uygun **undersampling/oversampling** teknikleri kullanılarak veri dengelenmelidir.
- `step` değişkeni saatlik periyotlara çevrilerek modele anlamlı özellikler kazandırılabilir.
- Yalnızca belirli türler üzerinden bir modelleme yapmak veya işlem türüne özel modeller eğitmek değerlendirilebilir.
